# Visualization Utilities Test Notebook

This notebook is a lightweight demo for the functions in `src/visualize.py`.

It is meant to help teammates understand:
- what each visualization function does,
- what kind of inputs it expects,
- what the output should look like,
- how to reuse the same functions later in evaluation and demo notebooks.

> Recommended use: local Jupyter / VS Code notebook now, Colab later if needed.


## 1. Notebook structure

We test the following functions one by one:

1. `plot_point_cloud(points)` → single-color 3D point cloud
2. `plot_point_cloud(points, colors=...)` → RGB-colored point cloud
3. `plot_point_cloud(points, labels=...)` → point cloud colored by class labels
4. `plot_heatmap(points, scores, query_text)` → continuous score heatmap
5. `plot_heatmap(..., top_percent=...)` → highlight only the strongest points
6. `plot_class_comparison(points, pred_labels, gt_labels)` → prediction vs ground-truth side by side
7. `save_figure(fig, path)` → export HTML + PNG


In [1]:
from pathlib import Path
import sys

# notebook path: <project_root>/notebooks/visualization_test.ipynb
PROJECT_ROOT = Path.cwd().resolve().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("src exists:", (PROJECT_ROOT / "src").exists())

import numpy as np

from src.visualize import (
    plot_point_cloud,
    plot_heatmap,
    plot_class_comparison,
    save_figure,
)


Project root: /Users/matte/Desktop/corsi_ghent/DeepLearning/ENV/Deep_learning_project-main
src exists: True


## 2. Synthetic test data

We generate a small artificial 3D point cloud made of 3 Gaussian clusters.

Why synthetic data first?
- it is fast,
- reproducible,
- easy to reason about,
- useful for smoke testing before using real S3DIS / Polycam data.


In [2]:
def make_synthetic_point_cloud(n_per_cluster: int = 800, seed: int = 42):
    rng = np.random.default_rng(seed)

    means = [
        np.array([0.0, 0.0, 0.0]),
        np.array([3.0, 0.0, 1.0]),
        np.array([1.5, 2.5, 2.0]),
    ]

    covs = [
        np.array([[0.20, 0.00, 0.00], [0.00, 0.15, 0.00], [0.00, 0.00, 0.10]]),
        np.array([[0.25, 0.05, 0.00], [0.05, 0.20, 0.02], [0.00, 0.02, 0.15]]),
        np.array([[0.18, -0.03, 0.00], [-0.03, 0.18, 0.04], [0.00, 0.04, 0.12]]),
    ]

    point_list = []
    label_list = []

    for class_id, (mean, cov) in enumerate(zip(means, covs)):
        cluster_points = rng.multivariate_normal(mean=mean, cov=cov, size=n_per_cluster)
        point_list.append(cluster_points)
        label_list.append(np.full(n_per_cluster, class_id, dtype=np.int32))

    points = np.vstack(point_list).astype(np.float32)
    labels = np.concatenate(label_list)
    return points, labels


def make_rgb_colors(points: np.ndarray) -> np.ndarray:
    mins = points.min(axis=0, keepdims=True)
    maxs = points.max(axis=0, keepdims=True)
    norm_points = (points - mins) / (maxs - mins + 1e-8)
    return norm_points.astype(np.float32)


def make_heatmap_scores(points: np.ndarray, query_center: np.ndarray) -> np.ndarray:
    distances = np.linalg.norm(points - query_center[None, :], axis=1)
    scores = np.exp(-(distances ** 2) / (2.0 * 0.9 ** 2))
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    return scores.astype(np.float32)


def make_noisy_predictions(gt_labels: np.ndarray, flip_probability: float = 0.12, n_classes: int = 3, seed: int = 123):
    rng = np.random.default_rng(seed)
    pred_labels = gt_labels.copy()
    flip_mask = rng.random(gt_labels.shape[0]) < flip_probability
    random_labels = rng.integers(0, n_classes, size=flip_mask.sum())
    pred_labels[flip_mask] = random_labels
    return pred_labels.astype(np.int32)


points, labels = make_synthetic_point_cloud(n_per_cluster=1000, seed=42)
rgb_colors = make_rgb_colors(points)
scores = make_heatmap_scores(points, query_center=np.array([1.5, 2.5, 2.0], dtype=np.float32))
pred_labels = make_noisy_predictions(labels, flip_probability=0.12, n_classes=3, seed=123)

print('points shape:', points.shape)
print('labels shape:', labels.shape)
print('rgb shape:', rgb_colors.shape)
print('scores shape:', scores.shape)
print('pred_labels shape:', pred_labels.shape)


points shape: (3000, 3)
labels shape: (3000,)
rgb shape: (3000, 3)
scores shape: (3000,)
pred_labels shape: (3000,)


## 3. Basic point cloud (single default color)

Use this when you only want to inspect the geometry of the point cloud,
without caring yet about RGB colors, labels, or model scores.


In [3]:
fig_plain = plot_point_cloud(points, title='Synthetic Point Cloud - No Labels')
fig_plain.show()


## 4. RGB-colored point cloud

Use this when the scene has RGB information and you want a more natural-looking view.
This is useful later for real scans where color helps interpretation.


In [4]:
fig_rgb = plot_point_cloud(points, colors=rgb_colors, title='Synthetic Point Cloud - RGB Colors')
fig_rgb.show()


## 5. Point cloud colored by semantic labels

Use this for segmentation-style visualizations.
Each class is assigned a discrete color, and the legend shows the class IDs.


In [5]:
fig_labels = plot_point_cloud(points, labels=labels, title='Synthetic Point Cloud - With Labels')
fig_labels.show()


## 6. Heatmap from continuous scores

Use this for open-vocabulary search or any per-point score.
Example: cosine similarity between the projected 3D features and a text query embedding.

Interpretation:
- cold colors → low relevance
- warm colors → high relevance


In [6]:
fig_heatmap = plot_heatmap(
    points,
    scores=scores,
    query_text='synthetic target',
    title='Heatmap for query: "synthetic target"',
)
fig_heatmap.show()


## 7. Heatmap with top-percent highlighting

Use this when the full heatmap is too dense or noisy.
The background points remain visible, but the strongest region is emphasized.

This is often more presentation-friendly than the fully dense heatmap.


In [7]:
fig_heatmap_top = plot_heatmap(
    points,
    scores=scores,
    query_text='synthetic target (top 5%)',
    title='Heatmap for query: "synthetic target" - Top 5%',
    top_percent=5.0,
)
fig_heatmap_top.show()


## 8. Predicted labels vs ground-truth labels

Use this for evaluation/debugging.
Left side = predicted labels, right side = ground truth.

This helps answer questions like:
- where does the model make mistakes?
- are the errors localized or spread everywhere?
- are specific classes systematically confused?


In [8]:
fig_comparison = plot_class_comparison(
    points,
    pred_labels=pred_labels,
    gt_labels=labels,
    title='Prediction vs Ground Truth - Synthetic Example',
)
fig_comparison.show()


## 9. Save a figure to disk

The utility saves:
- an interactive `.html` file
- a static `.png` file for slides or reports


In [9]:
saved_paths = save_figure(fig_labels, 'results/figures/synthetic_labels_test')
saved_paths


{'html': PosixPath('results/figures/synthetic_labels_test.html'),
 'png': PosixPath('results/figures/synthetic_labels_test.png')}